##Project Goal:
1. Baseline Model: lyric text only
2. controlled model: same lyric text, but control tokens added
3. Research Question: Does adding control tokens help the model generate lyrics that better match the requested style?


In [ ]:
#imports
#!pip install transformers datasets evaluate accelerate #hugging face import

import pandas as pd
import numpy as np
import math
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer)


##Data Loading & Cleaning
data found on github:https://github.com/walkerkq/musiclyrics/blob/master/billboard_lyrics_1964-2015.csv



In [ ]:
import pandas as pd
import re

# load and inspect
df = pd.read_csv("/content/billboard_lyrics_1964-2015.csv", encoding="latin1")

# make columns lowercase once and for all
df.columns = df.columns.str.lower()

print(df.columns)
print(df.shape)
display(df.head())

# drop rows with missing/empty lyrics
df = df.dropna(subset=["lyrics"]).copy()
df["lyrics"] = df["lyrics"].astype(str).str.strip()
df = df[df["lyrics"] != ""]

# remove placeholder rows
bad_phrases = ["instrumental", "lyrics unavailable", "no lyrics"]
df = df[~df["lyrics"].str.lower().isin(bad_phrases)]

# standardize text fields
def clean_basic(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["song"] = df["song"].apply(clean_basic)
df["artist"] = df["artist"].apply(clean_basic)

# create cleaned artist column for duplicate matching
def normalize_artist(text):
    text = text.lower()
    text = re.sub(r"\b(feat\.?|featuring|with|and)\b.*", "", text)
    text = re.sub(r"[^a-z0-9 ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["artist_clean"] = df["artist"].apply(normalize_artist)

# clean lyric text
def clean_lyrics(text):
    text = str(text)

    # remove bracketed section labels like [Verse], [Chorus]
    text = re.sub(r"\[.*?\]", "", text)

    # remove simple repeat markers / parenthetical metadata
    text = re.sub(r"\((x\d+|repeat.*?)\)", "", text, flags=re.IGNORECASE)

    # normalize curly quotes/apostrophes
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')

    # collapse spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # collapse repeated blank lines
    text = re.sub(r"\n\s*\n+", "\n", text)

    return text.strip()

df["lyrics_clean"] = df["lyrics"].apply(clean_lyrics)

# remove duplicates
def normalize_lyrics_for_match(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\n]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["lyrics_norm"] = df["lyrics_clean"].apply(normalize_lyrics_for_match)

df = df.drop_duplicates(subset=["song", "artist_clean", "year"])
df = df.drop_duplicates(subset=["song", "artist_clean", "lyrics_norm"])

# filter out unusable songs
df["word_count"] = df["lyrics_clean"].str.split().str.len()
df = df[(df["word_count"] >= 40) & (df["word_count"] <= 500)].copy()

print("After cleaning/filtering:", df.shape)

# split each song lyric into smaller chunks
rows = []

for _, row in df.iterrows():
    chunks = [c.strip() for c in row["lyrics_clean"].split("\n") if c.strip()]

    for i in range(0, len(chunks), 4):
        chunk = "\n".join(chunks[i:i+4]).strip()

        if len(chunk.split()) >= 20:
            rows.append({
                "song": row["song"],
                "artist": row["artist"],
                "year": row["year"],
                "rank": row["rank"],
                "source": row["source"],
                "lyrics_chunk": chunk
            })

chunk_df = pd.DataFrame(rows)

print("Chunked dataset shape:", chunk_df.shape)
display(chunk_df.head())

def assign_decade(year):
    year = int(year)
    if 1965 <= year <= 1969:
        return "1960s"
    elif 1970 <= year <= 1979:
        return "1970s"
    elif 1980 <= year <= 1989:
        return "1980s"
    elif 1990 <= year <= 1999:
        return "1990s"
    elif 2000 <= year <= 2009:
        return "2000s"
    elif 2010 <= year <= 2015:
        return "2010s"
    else:
        return None

chunk_df["decade"] = chunk_df["year"].apply(assign_decade)

print(chunk_df["decade"].value_counts())
display(chunk_df.head())

#save as file
chunk_df.to_csv("/content/cleaned_lyrics_chunks.csv", index=False)

Index(['rank', 'song', 'artist', 'year', 'lyrics', 'source'], dtype='object')
(5100, 6)


,rank,song,artist,year,lyrics,source
0,1,wooly bully,sam the sham and the pharaohs,1965,sam the sham miscellaneous wooly bully wooly b...,3.0
1,2,i cant help myself sugar pie honey bunch,four tops,1965,sugar pie honey bunch you know that i love yo...,1.0
2,3,i cant get no satisfaction,the rolling stones,1965,,1.0
3,4,you were on my mind,we five,1965,when i woke up this morning you were on my mi...,1.0
4,5,youve lost that lovin feelin,the righteous brothers,1965,you never close your eyes anymore when i kiss...,1.0


After cleaning/filtering: (3942, 10)
Chunked dataset shape: (3942, 6)


,song,artist,year,rank,source,lyrics_chunk
0,wooly bully,sam the sham and the pharaohs,1965,1,3.0,sam the sham miscellaneous wooly bully wooly b...
1,i cant help myself sugar pie honey bunch,four tops,1965,2,1.0,sugar pie honey bunch you know that i love you...
2,you were on my mind,we five,1965,4,1.0,when i woke up this morning you were on my min...
3,youve lost that lovin feelin,the righteous brothers,1965,5,1.0,you never close your eyes anymore when i kiss ...
4,downtown,petula clark,1965,6,1.0,when youre alone and life is making you lonely...


decade
1980s    924
1970s    896
1990s    722
2000s    543
1960s    453
2010s    404
Name: count, dtype: int64


,song,artist,year,rank,source,lyrics_chunk,decade
0,wooly bully,sam the sham and the pharaohs,1965,1,3.0,sam the sham miscellaneous wooly bully wooly b...,1960s
1,i cant help myself sugar pie honey bunch,four tops,1965,2,1.0,sugar pie honey bunch you know that i love you...,1960s
2,you were on my mind,we five,1965,4,1.0,when i woke up this morning you were on my min...,1960s
3,youve lost that lovin feelin,the righteous brothers,1965,5,1.0,you never close your eyes anymore when i kiss ...,1960s
4,downtown,petula clark,1965,6,1.0,when youre alone and life is making you lonely...,1960s


##Data Clean-up Continued.
I cleaned up the csv file manually in google sheets and inputted the mood based on the lyric-chunk.


In [ ]:
import re

df = pd.read_csv("/content/lyrics_final.csv")

#fix encoding artifacts
def fix_encoding_text(text):
    text = str(text)
    text = text.replace("Ì¢", "'")
    text = text.replace("Ã¢", "'")
    text = text.replace("Ã", "")
    text = text.replace("�", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["lyrics_chunk"] = df["lyrics_chunk"].apply(fix_encoding_text)

print(df.columns)
print(df.shape)
df.head()

df.to_csv("/content/lyrics_model_ready.csv", index=False)

Index(['lyrics_chunk', 'song', 'artist', 'year', 'rank', 'source', 'decade',
       'mood'],
      dtype='object')
(239, 8)


##Split train, validation, test by song


In [ ]:
from sklearn.model_selection import train_test_split
df = pd.read_csv("/content/lyrics_model_ready.csv")

#split by each unique song
songs = df["song"].drop_duplicates()

train_songs, temp_songs = train_test_split(songs, test_size=0.3, random_state=42)
val_songs, test_songs = train_test_split(temp_songs, test_size=0.5, random_state=42)

train_df = df[df["song"].isin(train_songs)].copy()
val_df = df[df["song"].isin(val_songs)].copy()
test_df = df[df["song"].isin(test_songs)].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

# create model text formats for ALL splits
for split_df in [train_df, val_df, test_df]:
    split_df["baseline_text"] = split_df["lyrics_chunk"]

    split_df["controlled_text"] = (
        "<MOOD=" + split_df["mood"].astype(str) + "> "
        + "<DECADE=" + split_df["decade"].astype(str) + ">\n"
        + split_df["lyrics_chunk"].astype(str)
    )

# now save AFTER creating the text columns
train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/val.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)

# quick check
display(train_df[["mood", "decade", "baseline_text", "controlled_text"]].head(3))

train: (167, 8)
val: (36, 8)
test: (36, 8)


,mood,decade,baseline_text,controlled_text
0,confident,1960s,you never close your eyes anymore when i kiss ...,<MOOD=confident> <DECADE=1960s>\nyou never clo...
1,confident,1960s,i said 'shotgun shoot em for he runs now do th...,<MOOD=confident> <DECADE=1960s>\ni said 'shotg...
2,confident,1960s,come here sister papas in the swing he aint to...,<MOOD=confident> <DECADE=1960s>\ncome here sis...


In [ ]:
print(set(train_df["song"]).intersection(set(val_df["song"])))
print(set(train_df["song"]).intersection(set(test_df["song"])))
print(set(val_df["song"]).intersection(set(test_df["song"])))

set()
set()
set()


##Training the baseline model using DistilGPT-2

In [ ]:
#imports
from transformers import (AutoTokenizer,
                          AutoModelForCausalLM,
                          DataCollatorForLanguageModeling,
                          TrainingArguments,
                          Trainer)

In [ ]:
#load the split files
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/val.csv")
test_df = pd.read_csv("/content/test.csv")

print(train_df.columns)
train_df.head()



Index(['lyrics_chunk', 'song', 'artist', 'year', 'rank', 'source', 'decade',
       'mood', 'baseline_text', 'controlled_text'],
      dtype='object')


,lyrics_chunk,song,artist,year,rank,source,decade,mood,baseline_text,controlled_text
0,you never close your eyes anymore when i kiss ...,youve lost that lovin feelin,the righteous brothers,1965,5,1,1960s,confident,you never close your eyes anymore when i kiss ...,<MOOD=confident> <DECADE=1960s>\nyou never clo...
1,i said 'shotgun shoot em for he runs now do th...,shotgun,junior walker the all stars,1965,15,3,1960s,confident,i said 'shotgun shoot em for he runs now do th...,<MOOD=confident> <DECADE=1960s>\ni said 'shotg...
2,come here sister papas in the swing he aint to...,papas got a brand new bag,james brown,1965,33,1,1960s,confident,come here sister papas in the swing he aint to...,<MOOD=confident> <DECADE=1960s>\ncome here sis...
3,once upon a time you dressed so fine threw the...,like a rolling stone,bob dylan,1965,41,1,1960s,confident,once upon a time you dressed so fine threw the...,<MOOD=confident> <DECADE=1960s>\nonce upon a t...
4,here they come again mmm catch us if you can m...,catch us if you can,the dave clark five,1965,54,3,1960s,confident,here they come again mmm catch us if you can m...,<MOOD=confident> <DECADE=1960s>\nhere they com...


In [ ]:
#baseline datasets
baseline_train = Dataset.from_pandas(train_df[["baseline_text"]].rename(columns={"baseline_text": "text"}))
baseline_val = Dataset.from_pandas(val_df[["baseline_text"]].rename(columns={"baseline_text": "text"}))
baseline_test = Dataset.from_pandas(test_df[["baseline_text"]].rename(columns={"baseline_text": "text"}))

In [ ]:
#load distilgpt-2 tokenizer and load model
model_name = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token=tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#tokenize text
def tokenize_fn (examples):
  return tokenizer(examples["text"])

tokenized_train = baseline_train.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_val = baseline_val.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_test = baseline_test.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/167 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

In [ ]:
#grouping into chunks
block_size = 128

def group_texts(examples):
  concatenated = {k: sum(examples[k], []) for k in examples.keys()}
  total_length = len(concatenated["input_ids"])
  total_length = (total_length // block_size) * block_size
  result = {
      k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
      for k , t in concatenated.items()
  }
  result["labels"] = result["input_ids"].copy()
  return result

lm_train = tokenized_train.map(group_texts, batched = True)
lm_val = tokenized_val.map(group_texts, batched = True)
lm_test = tokenized_test.map(group_texts, batched = True)

Map:   0%|          | 0/167 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

In [ ]:
#collate data
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
#train setup
training_args = TrainingArguments(
    output_dir="/content/baseline_distilgpt2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=20,
    fp16=True,
    report_to="none"
)
#train baseline model
trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
    processing_class = tokenizer,
)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.984579,3.819785
2,3.697475,3.782557
3,3.455843,3.784420
4,3.451019,3.793942
5,3.414157,3.798142


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1060, training_loss=3.6147798214318616, metrics={'train_runtime': 111.5639, 'train_samples_per_second': 19.003, 'train_steps_per_second': 9.501, 'total_flos': 69243638906880.0, 'train_loss': 3.6147798214318616, 'epoch': 5.0})

In [ ]:
#eval baseline
eval_results = trainer.evaluate()
print(eval_results)

baseline_perplexity = math.exp(eval_results["eval_loss"])
print("Baseline perplexity:", baseline_perplexity)

{'eval_loss': 3.7981419563293457, 'eval_runtime': 0.5816, 'eval_samples_per_second': 153.022, 'eval_steps_per_second': 77.371, 'epoch': 5.0}
Baseline perplexity: 44.618204854151685


In [ ]:
#save baseline
trainer.save_model("/content/baseline_distilgpt2/final")
tokenizer.save_pretrained("/content/baseline_distilgpt2/final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/baseline_distilgpt2/final/tokenizer_config.json',
 '/content/baseline_distilgpt2/final/tokenizer.json')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_baseline = AutoTokenizer.from_pretrained("/content/baseline_distilgpt2/final")
model_baseline = AutoModelForCausalLM.from_pretrained("/content/baseline_distilgpt2/final").to(device)
model_baseline.eval()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
#generate baseline samples
prompt = "write short song lyrics\n"
inputs = tokenizer_baseline(prompt, return_tensors="pt").to(device)

output = model_baseline.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    top_k=40,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3
)

print(tokenizer_baseline.decode(output[0], skip_special_tokens=True))

write short song lyrics
Theres no time to go the party that i used all my life and then when it came out its been a lot of fun you dont give up cause thats so much trouble now ifi cant find anything ive gotta be honest but im still in love with this place ill tell me about every one


In [ ]:
#save baseline sample results
baseline_eval = trainer.evaluate()
baseline_perplexity = math.exp(baseline_eval["eval_loss"])

print("Baseline eval loss:", baseline_eval["eval_loss"])
print("Baseline perplexity:", baseline_perplexity)

Baseline eval loss: 3.7981419563293457
Baseline perplexity: 44.618204854151685


##Building the Controlled Datasets Using DistilGPT-2



In [ ]:
from datasets import Dataset
import pandas as pd

train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/val.csv")
test_df = pd.read_csv("/content/test.csv")

controlled_train = Dataset.from_pandas(
    train_df[["controlled_text"]].rename(columns={"controlled_text": "text"})
)
controlled_val = Dataset.from_pandas(
    val_df[["controlled_text"]].rename(columns={"controlled_text": "text"})
)
controlled_test = Dataset.from_pandas(
    test_df[["controlled_text"]].rename(columns={"controlled_text": "text"})
)

In [ ]:
#reload distilgpt
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#inserting my control tokens
special_tokens = {
    "additional_special_tokens": [
        "<MOOD=happy>",
        "<MOOD=sad>",
        "<MOOD=reflective>",
        "<MOOD=confident>",
        "<DECADE=1960s>",
        "<DECADE=1970s>",
        "<DECADE=1980s>",
        "<DECADE=1990s>",
        "<DECADE=2000s>",
        "<DECADE=2010s>",
    ]
}

num_added = tokenizer.add_special_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))

print("Added tokens:", num_added)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Added tokens: 10


In [ ]:
#tokenize the controlled text
def tokenize_fn(examples):
    return tokenizer(examples["text"])

tokenized_train = controlled_train.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_val   = controlled_val.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_test  = controlled_test.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/167 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

In [ ]:
block_size = 128

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_train = tokenized_train.map(group_texts, batched=True)
lm_val   = tokenized_val.map(group_texts, batched=True)
lm_test  = tokenized_test.map(group_texts, batched=True)

Map:   0%|          | 0/167 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

In [ ]:
#data collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from transformers import TrainingArguments
#set training arguments
training_args = TrainingArguments(
    output_dir="/content/controlled_distilgpt2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=20,
    fp16=True,
    report_to="none"
)

In [ ]:
#training the controlled model
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss
1,3.861386,3.823810
2,3.577139,3.791709
3,3.591234,3.778958
4,3.509466,3.781747
5,3.546579,3.786338


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1075, training_loss=3.629773534730423, metrics={'train_runtime': 245.1817, 'train_samples_per_second': 8.749, 'train_steps_per_second': 4.385, 'total_flos': 70060191252480.0, 'train_loss': 3.629773534730423, 'epoch': 5.0})

In [ ]:
#evaluate controlled model
import math

controlled_eval = trainer.evaluate()
controlled_perplexity = math.exp(controlled_eval["eval_loss"])

print("Controlled eval loss:", controlled_eval["eval_loss"])
print("Controlled perplexity:", controlled_perplexity)

Controlled eval loss: 3.7863378524780273
Controlled perplexity: 44.094623215914744


In [ ]:
#save controlled model
trainer.save_model("/content/controlled_distilgpt2/final")
tokenizer.save_pretrained("/content/controlled_distilgpt2/final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/controlled_distilgpt2/final/tokenizer_config.json',
 '/content/controlled_distilgpt2/final/tokenizer.json')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_controlled = AutoTokenizer.from_pretrained("/content/controlled_distilgpt2/final")
model_controlled = AutoModelForCausalLM.from_pretrained("/content/controlled_distilgpt2/final").to(device)
model_controlled.eval()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50267, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50267, bias=False)
)

In [ ]:
# controlled model generation
prompt = "<MOOD=happy> <DECADE=2010s>\n"
inputs = tokenizer_controlled(prompt, return_tensors="pt").to(device)

output = model_controlled.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    top_k=40,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3
)

print(tokenizer_controlled.decode(output[0], skip_special_tokens=False))

<MOOD=happy> <DECADE=2010s>
i think the song is a bit like that but i still cant find out why it comes to me and so many people dont know about you then they say nothing ill be around for some time when its my last show all goneand im alone oh no one can help make this seem real yeahcause how


In [ ]:
model_controlled.eval()

prompts = [
    "<MOOD=happy> <DECADE=1980s>\n",
    "<MOOD=sad> <DECADE=1990s>\n",
    "<MOOD=reflective> <DECADE=1970s>\n",
    "<MOOD=confident> <DECADE=2000s>\n",
]

controlled_samples = []

for prompt in prompts:
    inputs = tokenizer_controlled(prompt, return_tensors="pt").to(device)

    output = model_controlled.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.8,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3
    )

    text = tokenizer_controlled.decode(output[0], skip_special_tokens=False)
    controlled_samples.append({"prompt": prompt, "generated_text": text})
    print("=" * 80)
    print(text)

<MOOD=happy> <DECADE=1980s>
theres been a long day and now i want to show you that weve got the right things here but im tired of being alone so let me tell ya about how much love your body makes for an interesting thing with all my friends on earth who are just as successful at it this time in our
<MOOD=sad> <DECADE=1990s>
i came to love my mom but i cant take it away from you and now this is what a sweet day we were all used for the dream come true that she told me how beautiful our hearts are so filled with joys like happiness when her heart breaks down your tears cause why can't they be
<MOOD=reflective> <DECADE=1970s>
bam i was really lost in the music of a million years ago now im just going mad at you all for what they say and why didnt we do it again with its your turn to make my way outta town tonight looking around id dont see that long till after this songyou can dance like
<MOOD=confident> <DECADE=2000s>
i got the feeling that i was in denial because of how you and my famil

In [ ]:
import pandas as pd

controlled_samples_df = pd.DataFrame(controlled_samples)
controlled_samples_df.to_csv("/content/controlled_generations.csv", index=False)
controlled_samples_df.head()

,prompt,generated_text
0,<MOOD=happy> <DECADE=1980s>\n,<MOOD=happy> <DECADE=1980s>\ntheres been a lon...
1,<MOOD=sad> <DECADE=1990s>\n,<MOOD=sad> <DECADE=1990s>\ni came to love my m...
2,<MOOD=reflective> <DECADE=1970s>\n,<MOOD=reflective> <DECADE=1970s>\nbam i was re...
3,<MOOD=confident> <DECADE=2000s>\n,<MOOD=confident> <DECADE=2000s>\ni got the fee...


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_baseline = AutoTokenizer.from_pretrained("/content/baseline_distilgpt2/final")
model_baseline = AutoModelForCausalLM.from_pretrained("/content/baseline_distilgpt2/final").to(device)


model_baseline.eval()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
baseline_samples = []

for i in range(4):
    prompt = "write short song lyrics\n"
    inputs = tokenizer_baseline(prompt, return_tensors="pt").to(model_baseline.device)

    output = model_baseline.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.8,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3
    )

    text = tokenizer_baseline.decode(output[0], skip_special_tokens=True)
    baseline_samples.append({"prompt": prompt, "generated_text": text})

baseline_samples_df = pd.DataFrame(baseline_samples)
baseline_samples_df.to_csv("/content/baseline_generations.csv", index=False)

In [ ]:
results_df = pd.DataFrame([
    {"model": "baseline", "eval_loss": baseline_eval["eval_loss"], "perplexity": baseline_perplexity},
    {"model": "controlled", "eval_loss": controlled_eval["eval_loss"], "perplexity": controlled_perplexity},
])

results_df.to_csv("/content/model_metrics.csv", index=False)
results_df

,model,eval_loss,perplexity
0,baseline,3.798142,44.618205
1,controlled,3.786338,44.094623


In [ ]:
import pandas as pd

baseline_df = pd.read_csv("/content/baseline_generations.csv")
controlled_df = pd.read_csv("/content/controlled_generations.csv")

n = min(len(baseline_df), len(controlled_df))

comparison_df = pd.DataFrame({
    "baseline_prompt": baseline_df.iloc[:n]["prompt"].values,
    "baseline_output": baseline_df.iloc[:n]["generated_text"].values,
    "controlled_prompt": controlled_df.iloc[:n]["prompt"].values,
    "controlled_output": controlled_df.iloc[:n]["generated_text"].values,
})

comparison_df.to_csv("/content/comparison_outputs.csv", index=False)
comparison_df.head()

,baseline_prompt,baseline_output,controlled_prompt,controlled_output
0,write short song lyrics\n,write short song lyrics\nby the way i was just...,<MOOD=happy> <DECADE=1980s>\n,<MOOD=happy> <DECADE=1980s>\ntheres been a lon...
1,write short song lyrics\n,write short song lyrics\nthe sound i was doing...,<MOOD=sad> <DECADE=1990s>\n,<MOOD=sad> <DECADE=1990s>\ni came to love my m...
2,write short song lyrics\n,write short song lyrics\ndont want to make you...,<MOOD=reflective> <DECADE=1970s>\n,<MOOD=reflective> <DECADE=1970s>\nbam i was re...
3,write short song lyrics\n,write short song lyrics\nwhat i wanna hear all...,<MOOD=confident> <DECADE=2000s>\n,<MOOD=confident> <DECADE=2000s>\ni got the fee...


In [ ]:
human_eval_df = pd.DataFrame({
    "prompt": comparison_df["controlled_prompt"],
    "baseline_output": comparison_df["baseline_output"],
    "controlled_output": comparison_df["controlled_output"],
    "baseline_coherence": "",
    "controlled_coherence": "",
    "baseline_style_match": "",
    "controlled_style_match": "",
    "notes": ""
})

human_eval_df.to_csv("/content/human_eval_template.csv", index=False)
human_eval_df.head()

,prompt,baseline_output,controlled_output,baseline_coherence,controlled_coherence,baseline_style_match,controlled_style_match,notes
0,<MOOD=happy> <DECADE=1980s>\n,write short song lyrics\nby the way i was just...,<MOOD=happy> <DECADE=1980s>\ntheres been a lon...,,,,,
1,<MOOD=sad> <DECADE=1990s>\n,write short song lyrics\nthe sound i was doing...,<MOOD=sad> <DECADE=1990s>\ni came to love my m...,,,,,
2,<MOOD=reflective> <DECADE=1970s>\n,write short song lyrics\ndont want to make you...,<MOOD=reflective> <DECADE=1970s>\nbam i was re...,,,,,
3,<MOOD=confident> <DECADE=2000s>\n,write short song lyrics\nwhat i wanna hear all...,<MOOD=confident> <DECADE=2000s>\ni got the fee...,,,,,


In [ ]:
import os
import shutil

project_folder = "/content/final_project_submission"
os.makedirs(project_folder, exist_ok=True)

files_to_copy = [
    "/content/baseline_generations.csv",
    "/content/controlled_generations.csv",
    "/content/comparison_outputs.csv",
    "/content/human_eval_template.csv",
    "/content/lyrics_final.csv",
    "/content/lyrics_model_ready.csv",
    "/content/model_metrics.csv",
    "/content/test.csv",
    "/content/train.csv",
    "/content/val.csv",
]

for file_path in files_to_copy:
    if os.path.exists(file_path):
        shutil.copy(file_path, project_folder)

shutil.make_archive("/content/final_project_submission", "zip", project_folder)

'/content/final_project_submission.zip'